<a href="https://colab.research.google.com/github/PollyIva/DL_in_CV/blob/main/DeepSORT_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Jun 25 15:50:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install ultralytics
!pip install torchreid
!pip install motmetrics
!pip install gdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.5 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for torchreid: filename=torchreid-0.2.5-py3-none-any.whl size=144324 sha256=7e2047e13bed2d892babb45ebc92f3ac9e4e096b30e6f26735b065579424fe68
  Stored in directory: /root/.cache/pip/wheels/5c/86/ff/80a1b78a90df470cbb12c075bf189ad33f1a41a881cf9e9a09
Successfully built torchreid
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 7.6 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/nwojke/deep_sort.git

Cloning into 'deep_sort'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 167 (delta 4), reused 3 (delta 3), pack-reused 148 (from 2)
Receiving objects: 100% (167/167), 86.11 KiB | 722.00 KiB/s, done.
Resolving deltas: 100% (85/85), done.


In [ ]:
%cd deep_sort

/content/deep_sort


In [ ]:
!pip install -r requirements.txt

In [ ]:
%cd ..

import os
import sys
sys.path.append("deep_sort")
print("DeepSORT path added")

/content
DeepSORT path added


In [ ]:
import cv2
import torch
import numpy as np
import time
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

PyTorch: 2.11.0+cu128
CUDA available: True
Using device: cuda


In [ ]:
folders = [
    "videos",
    "results",
    "overlays",
    "metrics",
    "configs"
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
print("Folders created")

Folders created


#2 — Detectors

Наша цель — чтобы любой детектор возвращал одинаковый формат:

[
    [x1, y1, x2, y2, confidence],
    ...
]

Мы будем брать только класс person (class id = 0).

In [ ]:
#Базовый класс детектора

class Detector:
    def detect(self, frame):
        raise NotImplementedError

# YOLOv8 Detector

class YOLODetector(Detector):

    def __init__(
        self,
        model_name="yolov8n.pt",
        conf=0.5,
        device=DEVICE
    ):

        self.model = YOLO(model_name)
        self.conf = conf
        self.device = device
    def detect(self, frame):
        result = self.model(
            frame,
            conf=self.conf,
            device=self.device,
            verbose=False
        )[0]
        detections = []
        for box in result.boxes:
            cls = int(box.cls[0])
            # только люди
            if cls != 0:
                continue
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            conf = float(box.conf[0])
            detections.append(
                [
                    int(x1),
                    int(y1),
                    int(x2),
                    int(y2),
                    conf
                ]
            )
        return detections


# RT-DETR Detector

# В ultralytics RT-DETR имеет такой же интерфейс.--------------------------------------

class RTDETRDetector(Detector):
    def __init__(
        self,
        model_name="rtdetr-l.pt",
        conf=0.5,
        device=DEVICE
    ):
        self.model = YOLO(model_name)
        self.conf = conf
        self.device = device
    def detect(self, frame):
        result = self.model(
            frame,
            conf=self.conf,
            device=self.device,
            verbose=False
        )[0]
        detections = []
        for box in result.boxes:
            if int(box.cls[0]) != 0:
                continue
            x1, y1, x2, y2 = (
                box.xyxy[0]
                .cpu()
                .numpy()
            )
            detections.append([
                int(x1),
                int(y1),
                int(x2),
                int(y2),
                float(box.conf[0])
            ])
        return detections



In [ ]:
#  Фабрика детекторов

def create_detector(name):
    if name == "yolov8n":
        return YOLODetector(
            "yolov8n.pt",
            conf=0.5
        )
    elif name == "yolov8m":
        return YOLODetector(
            "yolov8m.pt",
            conf=0.5
        )
    elif name == "rtdetr":
        return RTDETRDetector(
            "rtdetr-l.pt",
            conf=0.5
        )
    else:
        raise ValueError(
            f"Unknown detector: {name}"
        )

In [ ]:
detector_name = "yolov8n"
detector = create_detector(detector_name)
print(
    "Loaded detector:",
    detector_name
)


# detector_name = "yolov8m"
# detector_name = "rtdetr"



Loaded detector: yolov8n


#3 — REID

In [ ]:
import torch
import torchreid
import torchvision.transforms as T
from PIL import Image
import numpy as np

/usr/local/lib/python3.12/dist-packages/torchreid/reid/metrics/rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [ ]:
class ReIDModel:
    def get_embedding(self, image):
        raise NotImplementedError

In [ ]:
class TorchReID(ReIDModel):

    def __init__(
        self,
        model_name,
        device=DEVICE
    ):
        self.device = device

        self.model = torchreid.models.build_model(
            name=model_name,
            num_classes=1000,
            pretrained=True
        )

        self.model.to(device)
        self.model.eval()

        self.transform = T.Compose([
            T.ToPILImage(),
            T.Resize((256, 128)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

    @torch.no_grad()
    def get_embedding(self, image):

        img = self.transform(image)

        img = img.unsqueeze(0).to(self.device)

        feature = self.model(img)

        feature = feature.cpu().numpy()[0]

        # L2 normalization
        feature /= np.linalg.norm(feature)

        return feature

In [ ]:
def create_reid(name):

    if name == "osnet_x0_25":
        return TorchReID(
            "osnet_x0_25"
        )

    elif name == "osnet_x1_0":
        return TorchReID(
            "osnet_x1_0"
        )

    elif name == "resnet50":
        return TorchReID(
            "resnet50"
        )

    else:
        raise ValueError(
            f"Unknown REID model: {name}"
        )

In [ ]:
reid_name = "osnet_x0_25"

reid = create_reid(reid_name)

print(
    "Loaded REID:",
    reid_name
)

Downloading...
From: https://drive.google.com/uc?id=1rb8UN5ZzPKRc_xvtHlyDh-cSz88YX9hs
To: /root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth
100%|██████████| 2.97M/2.97M [00:00<00:00, 18.7MB/s]


Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
Loaded REID: osnet_x0_25


#4 — DeepSORT.

In [ ]:
import sys
sys.path.append('./deep_sort')

from deep_sort import nn_matching
from deep_sort.detection import Detection
from deep_sort.tracker import Tracker

In [ ]:
class DeepSORTWrapper:

    def __init__(
        self,
        max_cosine_distance=0.3,
        nn_budget=100
    ):

        self.metric = nn_matching.NearestNeighborDistanceMetric(
            "cosine",
            max_cosine_distance,
            nn_budget
        )

        self.tracker = Tracker(self.metric)
    def update(
        self,
        frame,
        detections,
        reid
    ):
        """
        detections:
        [x1, y1, x2, y2, confidence]
        """

        ds_detections = []


        for det in detections:

            x1, y1, x2, y2, conf = det


            # Вырезаем человека
            crop = frame[y1:y2, x1:x2]


            if crop.size == 0:
                continue


            # Получаем embedding
            feature = reid.get_embedding(crop)


            # DeepSORT использует tlwh
            bbox = [
                x1,
                y1,
                x2 - x1,
                y2 - y1
            ]


            ds_detections.append(
                Detection(
                    bbox,
                    conf,
                    feature
                )
            )
                    # Предсказание нового положения треков
        self.tracker.predict()


        # Сопоставление детекций и треков
        self.tracker.update(
            ds_detections
        )


        results = []


        for track in self.tracker.tracks:

            if not track.is_confirmed():
                continue


            if track.time_since_update > 1:
                continue


            bbox = track.to_tlbr()


            results.append({
                "id": track.track_id,
                "bbox": bbox
            })


        return results

In [ ]:
tracker = DeepSORTWrapper(
    max_cosine_distance=0.3,
    nn_budget=100
)
detector = create_detector(
    "yolov8n"
)

reid = create_reid(
    "osnet_x0_25"
)

Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"


#5 — запуск трекинга и сохранение результатов

In [ ]:
class MOTWriter:
    def __init__(self, filename):
        self.file = open(filename, "w")


    def write(self, frame_id, track_id, bbox):
        x1, y1, x2, y2 = bbox

        width = x2 - x1
        height = y2 - y1

        line = (
            f"{frame_id},{track_id},"
            f"{x1:.2f},{y1:.2f},"
            f"{width:.2f},{height:.2f},"
            "1,-1,-1,-1\n"
        )

        self.file.write(line)


    def close(self):
        self.file.close()

In [ ]:
def draw_tracks(frame, tracks):

    for track in tracks:

        x1, y1, x2, y2 = map(int, track["bbox"])

        track_id = track["id"]

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (0, 255, 0),
            2
        )

        cv2.putText(
            frame,
            f"ID {track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )

    return frame

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "takshmandar/mot16-dataset"
)

print(path)

100%|██████████| 1.82G/1.82G [00:37<00:00, 52.7MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/takshmandar/mot16-dataset/versions/1


In [ ]:
import os

for root, dirs, files in os.walk(path):
    print(root, dirs)
    print(files[:5])
    break

/root/.cache/kagglehub/datasets/takshmandar/mot16-dataset/versions/1 ['train', 'test']
[]


In [ ]:
mot09 = os.path.join(path, "train", "MOT16-09")

for item in os.listdir(mot09):
    print(item)

det
gt
img1
seqinfo.ini


In [ ]:
img_dir = os.path.join(path, "train", "MOT16-09", "img1")

images = sorted(os.listdir(img_dir))
print("Количество кадров:", len(images))
print("Первые кадры:", images[:5])

Количество кадров: 525
Первые кадры: ['000001.jpg', '000002.jpg', '000003.jpg', '000004.jpg', '000005.jpg']


In [ ]:
def run_sequence(
    img_dir,
    output_video,
    mot_path,
    detector,
    reid,
    tracker,
    fps=30
):
    import cv2
    import time
    import os

    frames = sorted(os.listdir(img_dir))

    first_frame = cv2.imread(os.path.join(img_dir, frames[0]))
    h, w = first_frame.shape[:2]

    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    mot_writer = MOTWriter(mot_path)

    total_time = 0

    for i, frame_name in enumerate(frames, start=1):

        frame_path = os.path.join(img_dir, frame_name)
        frame = cv2.imread(frame_path)

        if frame is None:
            continue

        start = time.time()

        detections = detector.detect(frame)
        tracks = tracker.update(frame, detections, reid)

        elapsed = time.time() - start
        total_time += elapsed

        fps_now = 1 / elapsed if elapsed > 0 else 0

        cv2.putText(frame, f"FPS: {fps_now:.1f}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

        frame = draw_tracks(frame, tracks)

        for t in tracks:
            mot_writer.write(i, t["id"], t["bbox"])

        writer.write(frame)

        if i % 50 == 0:
            print(f"Frame {i}/{len(frames)} FPS={fps_now:.2f}")

    writer.release()
    mot_writer.close()

    avg_fps = len(frames) / total_time if total_time > 0 else 0

    print("DONE")
    print("AVG FPS:", avg_fps)

    return avg_fps

In [ ]:
avg_fps = run_sequence(
    img_dir=img_dir,
    output_video="results/MOT16-09.mp4",
    mot_path="results/MOT16-09.txt",
    detector=detector,
    reid=reid,
    tracker=tracker
)

Frame 50/525 FPS=7.34
Frame 100/525 FPS=5.38
Frame 150/525 FPS=7.50
Frame 200/525 FPS=5.53
Frame 250/525 FPS=5.90
Frame 300/525 FPS=8.67
Frame 350/525 FPS=8.93
Frame 400/525 FPS=8.32
Frame 450/525 FPS=4.87
Frame 500/525 FPS=5.76
DONE
AVG FPS: 6.830718064318189


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define a base directory in your Google Drive to save results
gdrive_output_dir = "/content/drive/MyDrive/deep_sort_results"
os.makedirs(gdrive_output_dir, exist_ok=True)
print(f"Google Drive output directory created: {gdrive_output_dir}")

Mounted at /content/drive
Google Drive output directory created: /content/drive/MyDrive/deep_sort_results


#6 — система перебора параметров

In [ ]:
import pandas as pd

experiment_results = []

def run_experiment(
    video_path,
    detector_name,
    reid_name,
    conf,
    cosine_distance,
    nn_budget
):

    print("="*50)
    print(
        f"Detector: {detector_name}, "
        f"REID: {reid_name}"
    )

    print(
        f"conf={conf}, "
        f"cosine={cosine_distance}"
    )


    detector = create_detector(
        detector_name
    )

    detector.conf = conf


    reid = create_reid(
        reid_name
    )


    tracker = DeepSORTWrapper(
        max_cosine_distance=cosine_distance,
        nn_budget=nn_budget
    )


    output_video = (
        "results/temp.mp4"
    )

    output_mot = (
        "results/temp.txt"
    )


    fps = run_sequence(
        img_dir=video_path,       # <-- переименовал аргумент для ясности
        output_video=output_video,
        mot_path=output_mot,
        detector=detector,
        reid=reid,
        tracker=tracker
    )


    result = {
        "detector": detector_name,
        "reid": reid_name,
        "confidence": conf,
        "cosine_distance": cosine_distance,
        "nn_budget": nn_budget,
        "fps": fps
    }


    experiment_results.append(result)


    return result

run_experiment(
    video_path=img_dir,

    detector_name="yolov8n",

    reid_name="osnet_x0_25",

    conf=0.5,

    cosine_distance=0.3,

    nn_budget=100
)


Detector: yolov8n, REID: osnet_x0_25
conf=0.5, cosine=0.3
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
Frame 50/525 FPS=6.80
Frame 100/525 FPS=8.75
Frame 150/525 FPS=7.83
Frame 200/525 FPS=5.96
Frame 250/525 FPS=6.07
Frame 300/525 FPS=8.41
Frame 350/525 FPS=6.00
Frame 400/525 FPS=8.14
Frame 450/525 FPS=6.69
Frame 500/525 FPS=6.13
DONE
AVG FPS: 6.8182638793539665


{'detector': 'yolov8n',
 'reid': 'osnet_x0_25',
 'confidence': 0.5,
 'cosine_distance': 0.3,
 'nn_budget': 100,
 'fps': 6.8182638793539665}

In [ ]:


detectors = [
    "yolov8n",
    "yolov8m",
    "rtdetr"
]


reids = [
    "osnet_x0_25",
    # "osnet_x1_0",
    # "resnet50"
]


confidences = [
    0.3,
    0.5,
    0.7
]


cosine_distances = [
#     0.2,
#     0.3,
    0.5
]


nn_budgets = [
    50,
    # 100
]

for detector in detectors:

    for reid in reids:

        for conf in confidences:

            for cosine in cosine_distances:

                for budget in nn_budgets:

                    run_experiment(
                        video_path=img_dir,
                        detector_name=detector,
                        reid_name=reid,
                        conf=conf,
                        cosine_distance=cosine,
                        nn_budget=budget
                    )

df = pd.DataFrame(
    experiment_results
)

df

Detector: yolov8n, REID: osnet_x0_25
conf=0.3, cosine=0.5
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
Frame 50/525 FPS=6.18
Frame 100/525 FPS=6.09
Frame 150/525 FPS=6.23
Frame 200/525 FPS=5.88
Frame 250/525 FPS=4.91
Frame 300/525 FPS=7.84
Frame 350/525 FPS=6.35
Frame 400/525 FPS=4.84
Frame 450/525 FPS=5.95
Frame 500/525 FPS=5.67
DONE
AVG FPS: 5.598900527186056
Detector: yolov8n, REID: osnet_x0_25
conf=0.5, cosine=0.5
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x0_25_imagenet.pth"
Frame 50/525 FPS=7.73
Frame 100/525 FPS=8.53
Frame 150/525 FPS=7.18
Frame 200/525 FPS=6.03
Frame 250/525 FPS=6.11
Frame 300/525 FPS=5.93
Frame 350/525 FPS=9.21
Frame 400/525 FPS=8.53
Frame 450/525 FPS=7.29
Frame 500/525 FPS=5.79
DONE
AVG FPS: 6.841211239557762
Detector: yolov8n, REID: osnet_x0_25
conf=0.7, cosine=0.5
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoin

,detector,reid,confidence,cosine_distance,nn_budget,fps
0,yolov8n,osnet_x0_25,0.5,0.3,100,6.818264
1,yolov8n,osnet_x0_25,0.3,0.5,50,5.598901
2,yolov8n,osnet_x0_25,0.5,0.5,50,6.841211
3,yolov8n,osnet_x0_25,0.7,0.5,50,9.384857
4,yolov8m,osnet_x0_25,0.3,0.5,50,4.369665
5,yolov8m,osnet_x0_25,0.5,0.5,50,4.700174
6,yolov8m,osnet_x0_25,0.7,0.5,50,5.767947
7,rtdetr,osnet_x0_25,0.3,0.5,50,3.131563
8,rtdetr,osnet_x0_25,0.5,0.5,50,3.856305
9,rtdetr,osnet_x0_25,0.7,0.5,50,4.416111


In [ ]:
# df.to_csv(
#     "metrics/experiments.csv",
#     index=False
# )


os.makedirs(os.path.join(gdrive_output_dir, "metrics"), exist_ok=True)
df.to_csv(
    os.path.join(gdrive_output_dir, "metrics", "experiments.csv"),
    index=False
)

7 — оценку качества трекинга

In [ ]:
!git clone https://github.com/JonathonLuiten/TrackEval.git
%cd TrackEval
!pip install -r requirements.txt
%cd ..

In [ ]:
import motmetrics as mm
import pandas as pd

def evaluate_mot(gt_file, pred_file):

    acc = mm.utils.compare_to_groundtruth(
        gt=mm.io.loadtxt(
            gt_file,
            fmt="mot15-2D"
        ),

        ts=mm.io.loadtxt(
            pred_file,
            fmt="mot15-2D"
        ),

        dist="iou",
        distth=0.5
    )


    metrics = [
        "mota",
        "idf1",
        "precision",
        "recall",
        "num_switches"
    ]


    mh = mm.metrics.create()


    summary = mh.compute(
        acc,
        metrics=metrics,
        name="tracking"
    )


    return summary

In [ ]:
result = evaluate_mot(
    "MOT16-09/gt/gt.txt",
    "results/MOT16-09.txt"
)

result

In [ ]:
import shutil


def prepare_trackeval(
    gt_path,
    prediction_path,
    sequence_name
):

    gt_dir = (
        "TrackEval/data/gt/"
        + sequence_name
    )

    pred_dir = (
        "TrackEval/data/trackers/our_tracker"
    )


    os.makedirs(
        gt_dir,
        exist_ok=True
    )

    os.makedirs(
        pred_dir,
        exist_ok=True
    )


    shutil.copy(
        gt_path,
        gt_dir + "/gt.txt"
    )


    shutil.copy(
        prediction_path,
        pred_dir + "/" + sequence_name + ".txt"
    )

In [ ]:
!python TrackEval/scripts/run_mot_challenge.py \
    --BENCHMARK MOT16 \
    --TRACKERS_TO_EVAL our_tracker \
    --METRICS HOTA